In [3]:
import os
import numpy 
import sys
import os

ROOT = os.path.abspath("..")  # adjust if needed
print(ROOT)
OPENPCDET_PATH = '/storage/project/r-gchou3-0/spanse30/OpenPCDet'
print(OPENPCDET_PATH)
# # Put OpenPCDet FIRST
sys.path.insert(0, OPENPCDET_PATH)

# # Import OpenPCDet pcdet
import pcdet
print("Using pcdet from:", pcdet.__file__)
import pickle as pkl

import os
from pathlib import Path
import copy
import numpy as np
import torch

from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.models import build_network, load_data_to_gpu
from pcdet.datasets import build_dataloader
from pcdet.utils import common_utils
import random

import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

import importlib
import helpers_ptt
import removal
importlib.reload(helpers_ptt)
importlib.reload(removal)
import torch.nn.functional as F

from collections import defaultdict

from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()


CFG_FILE_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml' 
CKPT_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_30.pth'  # <- put your ckpt here

cfg_from_yaml_file(CFG_FILE_MF, cfg)
cfg.TAG = Path(CFG_FILE_MF).stem
cfg.EXP_GROUP_PATH = 'centerpoint_waymo_demo'

import os
# cfg.DATA_CONFIG.DATA_PATH = os.path.join(OPENPCDET_PATH, 'data', 'waymo')

logger_mf = common_utils.create_logger()
logger_mf.info(f'Loaded cfg from {CFG_FILE_MF}')


dataset_mf, test_loader_mf, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger_mf,
    training=False
)

len_test_mf = len(dataset_mf)
logger_mf.info(f'Test set length: {len_test_mf}')
model_mf = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=dataset_mf
)

logger_mf.info(f'Loading checkpoint from: {CKPT_MF}')
model_mf.load_params_from_file(filename=CKPT_MF, logger=logger_mf, to_cpu=False)
model_mf.cuda()
model_mf.eval()

data_iter_mf = iter(test_loader_mf)
batch_dict_mf = next(data_iter_mf)

load_data_to_gpu(batch_dict_mf)


    

2026-04-06 17:29:33,254   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml
2026-04-06 17:29:33,254   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml
2026-04-06 17:29:33,255   INFO  Loading Waymo dataset
2026-04-06 17:29:33,255   INFO  Loading Waymo dataset


/storage/scratch1/9/spanse30
/storage/project/r-gchou3-0/spanse30/OpenPCDet
Using pcdet from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/pcdet/__init__.py


2026-04-06 17:29:35,060   INFO  Total skipped info 0
2026-04-06 17:29:35,060   INFO  Total skipped info 0
2026-04-06 17:29:35,061   INFO  Total samples for Waymo dataset: 39987
2026-04-06 17:29:35,061   INFO  Total samples for Waymo dataset: 39987
2026-04-06 17:29:35,062   INFO  Test set length: 39987
2026-04-06 17:29:35,062   INFO  Test set length: 39987
2026-04-06 17:29:35,415   INFO  Loading checkpoint from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_30.pth
2026-04-06 17:29:35,415   INFO  Loading checkpoint from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_30.pth
2026-04-06 17:29:35,421   INFO  ==> Loading parameters from checkpoint /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_30.pth to GPU
2026-04-06 17:29:35,421   INFO  =

In [2]:
def generate_timing_classification(target_frames, window_size):
    classification = []

    keys = sorted(target_frames.keys())
    n_prev = window_size - 1

    for k in keys:
        cur = target_frames.get(k)

        prev_vals = [target_frames.get(k - i) for i in range(1, n_prev + 1)]

        prev_all_none = all(v is None for v in prev_vals)
        prev_any_not_none = any(v is not None for v in prev_vals)

        if cur is None and prev_all_none:
            classification.append(0)

        elif cur is not None and prev_all_none:
            classification.append(1)

        elif cur is not None and prev_any_not_none:
            classification.append(2)

        elif cur is None and prev_any_not_none:
            classification.append(3)

    return classification

In [3]:
import time

In [8]:
frame_ids = [info['frame_id'] for info in dataset_sf.infos]

# Split the frame_id to get only the segment name (everything before the last underscore)
# and use set() to find unique values
unique_segments = sorted(list(set(fid.rsplit('_', 1)[0] for fid in frame_ids)))


modes = ["removal_20"]
for mode in modes:
    dataset_segs = os.listdir(f"/storage/scratch1/9/spanse30/PTT/datasets/{mode}")
    annos_segs = os.listdir(f"/storage/scratch1/9/spanse30/PTT/annos/{mode}")
    assert(len(unique_segments) == len(dataset_segs))
    assert(len(unique_segments) == len(annos_segs))
    start = time.perf_counter()
    for seg in unique_segments:
        filepath = f"/storage/scratch1/9/spanse30/PTT/datasets/{mode}/{seg}_d.pkl"
        with open(filepath, 'rb') as file:
            seg_data = pkl.load(file)
        target_frames = {}
        for i, frame in enumerate(seg_data):
            target_frames[i] = frame['spoof_gt']

        timing_msf8 = generate_timing_classification(target_frames, 8)

        filepath = f"/storage/scratch1/9/spanse30/PTT/annos/{mode}/{seg}_a.pkl"
        with open(filepath, 'rb') as file:
            annos_data = pkl.load(file)
        if('timing_msf' in annos_data):
            annos_data['timing_msf4'] = annos_data.pop('timing_msf')
            annos_data['timing_msf8'] = timing_msf8

            with open(filepath, 'wb') as file:
                pkl.dump(annos_data, file)
                
    end = time.perf_counter()
    print(f"finished adding msf 8 timing for mode {mode} in {end - start:.4f} s")

# print(f"{end - start:.4f}")

finished adding msf 8 timing for mode removal_20 in 598.0208 s


In [ ]:
modes = ["rel_fixed_easy", "rel_fixed_medium", "rel_fixed_hard",  "global_easy", "global_medium", "global_hard"]